# Get results out of my WandB automatically

In [1]:
import wandb
import numpy as np
from itertools import product
import pandas as pd

api = wandb.Api()
runs = api.runs("labeebah-islaam/world_models")

In [ ]:
modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]
metric = "actor_critic/eval/planned_cumulative_reward"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

# Filter runs that do NOT start with 'pruned'
runs = [r for r in runs if not r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched = []
    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner)
            ):
                val = get_last_value(run)
                if val is not None:
                    matched.append(val)
        except KeyError:
            continue

    if len(matched) >= 3:
        print(f"Matched {len(matched)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        arr = np.array(matched[:3])  # only take first 3
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": arr.mean(),
            "std": arr.std()
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": None,
            "std": None
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("sweep_results.csv", index=False)

       mode  planning_steps  threshold  inner_steps  mean   std
0    reward               0        2.0            0  None  None
1    reward               0        2.0            1  None  None
2    reward               0        2.0            2  None  None
3    reward               0        2.0            5  None  None
4    reward               0        1.5            0  None  None
..      ...             ...        ...          ...   ...   ...
163   value              20        1.5            5  None  None
164   value              20        1.0            0  None  None
165   value              20        1.0            1  None  None
166   value              20        1.0            2  None  None
167   value              20        1.0            5  None  None

[168 rows x 6 columns]


In [ ]:
import numpy as np
import pandas as pd
from itertools import product

modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]

metric = "actor_critic/eval/planned_cumulative_reward"
meta_metric = "meta_planning_depth"
eval_step_key = "eval_step"
num_planning_steps_key = "actor_critic/eval/num_planning_steps"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

def get_last_eval_step(run):
    """Episode length = last eval_step for the run."""
    try:
        hist = run.history(keys=[eval_step_key], pandas=True)
        if not hist.empty:
            vals = hist[eval_step_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_num_planning_steps(run):
    """Single scalar; prefer summary, fall back to last history value."""
    try:
        v = run.summary.get(num_planning_steps_key, None)
        if v is not None:
            return float(v)
    except Exception:
        pass
    try:
        hist = run.history(keys=[num_planning_steps_key], pandas=True)
        if not hist.empty:
            vals = hist[num_planning_steps_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_meta_depth_counts(run):
    """Counts of meta_planning_depth occurrences per run for depths 1..4."""
    try:
        hist = run.history(keys=[meta_metric], pandas=True)
        if not hist.empty:
            vals = hist[meta_metric].dropna().astype(int).values
            return {d: int(np.sum(vals == d)) for d in [1, 2, 3, 4]}
    except Exception:
        pass
    return {d: 0 for d in [1, 2, 3, 4]}

# Filter runs that DO start with 'pruned'
runs = [r for r in runs if r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched_records = []

    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            # enforce planning_depth condition
            required_depth = 3 if (step == 0 or inner == 0) else 5

            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner) and
                int(cfg["evaluation"]["planning_depth"]) == required_depth
            ):
                rec = {}
                rec["reward"] = get_last_value(run)
                rec["ep_len"] = get_last_eval_step(run)
                rec["meta_counts"] = get_meta_depth_counts(run)
                rec["num_planning_steps"] = get_num_planning_steps(run)

                # require reward & episode length to include the run
                if rec["reward"] is not None and rec["ep_len"] is not None:
                    matched_records.append(rec)
        except KeyError:
            continue

    if len(matched_records) >= 3:
        print(f"Matched {len(matched_records)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        recs = matched_records[:3]

        arr_vals = np.array([r["reward"] for r in recs], dtype=float)
        arr_lens = np.array([r["ep_len"] for r in recs], dtype=float)

        # meta-depth counts per run
        counts_per_run = {d: np.array([r["meta_counts"][d] for r in recs], dtype=float) for d in [1, 2, 3, 4]}
        counts_mean = {d: float(np.mean(counts_per_run[d])) for d in [1, 2, 3, 4]}
        counts_std  = {d: float(np.std(counts_per_run[d]))  for d in [1, 2, 3, 4]}

        # num_planning_steps per run (allow NaN)
        nps = np.array([r["num_planning_steps"] if r["num_planning_steps"] is not None else np.nan for r in recs], dtype=float)
        nps_mean = float(np.nanmean(nps)) if np.any(~np.isnan(nps)) else None
        nps_std  = float(np.nanstd(nps))  if np.any(~np.isnan(nps)) else None

        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": arr_vals.mean(),
            "reward_std": arr_vals.std(),
            "ep_length_mean": arr_lens.mean(),
            "ep_length_std": arr_lens.std(),
            "num_planning_steps_mean": nps_mean,
            "num_planning_steps_std": nps_std,
            **{f"meta_depth_{d}_mean": counts_mean[d] for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": counts_std[d] for d in [1, 2, 3, 4]},
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": None,
            "reward_std": None,
            "ep_length_mean": None,
            "ep_length_std": None,
            "num_planning_steps_mean": None,
            "num_planning_steps_std": None,
            **{f"meta_depth_{d}_mean": None for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": None for d in [1, 2, 3, 4]},
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("pruned_sweep_results.csv", index=False)

        mode  planning_steps reward_mean reward_std ep_length_mean  \
0  (reward,)              20        None       None           None   
1   (value,)              20        None       None           None   

  ep_length_std num_planning_steps_mean num_planning_steps_std  \
0          None                    None                   None   
1          None                    None                   None   

  meta_depth_1_mean meta_depth_2_mean meta_depth_3_mean meta_depth_4_mean  \
0              None              None              None              None   
1              None              None              None              None   

  meta_depth_1_std meta_depth_2_std meta_depth_3_std meta_depth_4_std  
0             None             None             None             None  
1             None             None             None             None  


# All games Ablation

In [17]:
import numpy as np
import pandas as pd

ALLOWED_MODES = {"reward", "value"}   # change if you want
TARGET_STEPS = 0                     # 0_roll only

def parse_run_name(name: str):
    """
    Expected pattern:
    <env>_<steps>_roll_<inner>_inner_<pp>_ent_<depth>_max_<mode>_seed_<seed>_time_...
    Example:
    UpNDown_0_roll_0_inner_0.05_ent_2_max_reward_seed_2_time_2026-01-25_02-09-23
    """
    if not name:
        return None
    parts = name.split("_")

    # allow optional prefix like "pruned"
    if parts[0] == "pruned":
        parts = parts[1:]

    try:
        env_type = parts[0]
        planning_steps = int(parts[1])
        planning_mode = parts[9]
        seed = int(parts[11])
        return {
            "env_type": env_type,
            "planning_steps": planning_steps,
            "planning_mode": planning_mode,
            "seed": seed,
        }
    except Exception:
        return None

def get_cfg_env_type(run):
    env_type = (run.config.get("env") or {}).get("env_type", None)
    if env_type is not None:
        return env_type
    info = parse_run_name(run.name or "")
    return None if info is None else info["env_type"]

def get_cfg_mode(run):
    mode = (run.config.get("evaluation") or {}).get("planning_mode", None)
    if mode is not None:
        return mode
    info = parse_run_name(run.name or "")
    return None if info is None else info["planning_mode"]

def get_cfg_seed(run):
    seed = (run.config.get("common") or {}).get("seed", None)
    if seed is not None:
        return seed
    info = parse_run_name(run.name or "")
    return None if info is None else info["seed"]

def get_cfg_steps(run):
    steps = (run.config.get("evaluation") or {}).get("planning_steps", None)
    if steps is not None:
        return steps
    info = parse_run_name(run.name or "")
    return None if info is None else info["planning_steps"]

# ---- FILTER: finished + steps==0 + allowed mode ----
filtered = []
for r in runs:
    if r.state != "finished":
        continue

    steps = get_cfg_steps(r)
    if steps is None or int(steps) != TARGET_STEPS:
        continue

    mode = get_cfg_mode(r)
    if mode not in ALLOWED_MODES:
        continue

    env_type = get_cfg_env_type(r)
    if env_type is None:
        continue

    filtered.append(r)

print("FILTERED:", len(filtered))

# ---- GROUP: (env_type, mode), keep at most one per seed 0/1/2 ----
groups = {}  # (env_type, mode) -> {seed: run}
for r in filtered:
    env_type = get_cfg_env_type(r)
    mode = get_cfg_mode(r)
    seed = get_cfg_seed(r)
    if env_type is None or mode is None or seed is None:
        continue

    key = (env_type, mode)
    groups.setdefault(key, {})

    # keep first run per seed (0,1,2)
    if seed in [0, 1, 2] and seed not in groups[key]:
        groups[key][seed] = r

# ---- AGGREGATE ----
results = []
for (env_type, mode), by_seed in sorted(groups.items()):
    rs = list(by_seed.values())

    recs = []
    for r in rs:
        reward = get_last_value(r)
        ep_len = get_last_eval_step(r)
        nps = get_num_planning_steps(r)
        meta = get_meta_depth_counts(r)

        if reward is None or ep_len is None:
            continue

        recs.append((reward, ep_len, nps, meta))

    if not recs:
        continue

    rewards = np.array([x[0] for x in recs], float)
    eplens  = np.array([x[1] for x in recs], float)
    nps_arr = np.array([x[2] if x[2] is not None else np.nan for x in recs], float)
    meta_counts = {d: np.array([x[3][d] for x in recs], float) for d in [1, 2, 3, 4]}

    row = {
        "env_type": env_type,
        "mode": mode,
        "n_runs": len(recs),
        "reward_mean": float(rewards.mean()),
        "reward_std": float(rewards.std()),
        "ep_length_mean": float(eplens.mean()),
        "ep_length_std": float(eplens.std()),
        "num_planning_steps_mean": float(np.nanmean(nps_arr)) if np.any(~np.isnan(nps_arr)) else None,
        "num_planning_steps_std": float(np.nanstd(nps_arr)) if np.any(~np.isnan(nps_arr)) else None,
    }

    for d in [1, 2, 3, 4]:
        row[f"meta_depth_{d}_mean"] = float(meta_counts[d].mean())
        row[f"meta_depth_{d}_std"]  = float(meta_counts[d].std())

    results.append(row)

df = pd.DataFrame(results).sort_values(["env_type", "mode"]) if results else pd.DataFrame()
print(df)
df.to_csv("zero_roll_by_env_and_mode.csv", index=False)

FILTERED: 100
          env_type    mode  n_runs    reward_mean   reward_std  \
0            Alien  reward       3    1166.666667   233.285709   
1           Amidar  reward       3     229.333333    12.657892   
2          Assault  reward       3     731.000000    51.270524   
3          Asterix  reward       3    6033.333333  1463.633227   
4        BankHeist  reward       3       6.666667     4.714045   
5       BattleZone  reward       3     333.333333   471.404521   
6         Breakout  reward       3      93.333333    22.365648   
7   ChopperCommand  reward       3    1133.333333    94.280904   
8     CrazyClimber  reward       3  104233.333333  6832.438966   
9      DemonAttack  reward       3     738.333333   353.466956   
10         Freeway  reward       3      32.000000     0.816497   
11       Frostbite  reward       3     273.333333     4.714045   
12          Gopher  reward       3    5360.000000   572.247033   
13            Hero  reward       3    3165.000000     0.000000

In [ ]:
import numpy as np
import pandas as pd

TARGET_PP = 0.05
ALLOWED_MODES = {"reward", "value"}

def float_close(a, b, tol=1e-8):
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return False

def parse_run_name(name: str):
    """
    Expected pattern:
    <env>_<steps>_roll_<inner>_inner_<pp>_ent_<depth>_max_<mode>_seed_<seed>_time_...
    """
    if not name:
        return None
    parts = name.split("_")

    # allow optional prefix like "pruned"
    if parts[0] == "pruned":
        parts = parts[1:]

    try:
        env_type = parts[0]
        planning_steps = int(parts[1])
        inner_steps = int(parts[3])
        planning_percentage = float(parts[5])
        planning_depth = int(parts[7])
        planning_mode = parts[9]
        seed = int(parts[11])
        return {
            "env_type": env_type,
            "planning_steps": planning_steps,
            "inner_steps": inner_steps,
            "planning_percentage": planning_percentage,
            "planning_depth": planning_depth,
            "planning_mode": planning_mode,
            "seed": seed,
        }
    except Exception:
        return None

def get_cfg_env_type(run):
    # try config first
    env_type = (run.config.get("env") or {}).get("env_type", None)
    if env_type is not None:
        return env_type
    # fallback to name
    info = parse_run_name(run.name or "")
    return None if info is None else info["env_type"]

def get_cfg_mode(run):
    mode = (run.config.get("evaluation") or {}).get("planning_mode", None)
    if mode is not None:
        return mode
    info = parse_run_name(run.name or "")
    return None if info is None else info["planning_mode"]

def get_cfg_pp(run):
    pp = run.config.get("evaluation.planning_percentage", None)  # sometimes flat
    if pp is None:
        pp = (run.config.get("evaluation") or {}).get("planning_percentage", None)
    if pp is not None:
        return pp
    info = parse_run_name(run.name or "")
    return None if info is None else info["planning_percentage"]

def get_cfg_seed(run):
    seed = (run.config.get("common") or {}).get("seed", None)
    if seed is not None:
        return seed
    info = parse_run_name(run.name or "")
    return None if info is None else info["seed"]

# ---- FILTER ----
filtered = []
for r in runs:
    print(r.name)
    if r.state != "finished":
        continue
    pp = get_cfg_pp(r)
    if pp is None or not float_close(pp, TARGET_PP):
        continue
    mode = get_cfg_mode(r)
    if mode not in ALLOWED_MODES:
        continue
    env_type = get_cfg_env_type(r)
    if env_type is None:
        continue
    filtered.append(r)

# ---- GROUP (env_type, mode) and keep at most one per seed 0/1/2 ----
groups = {}
for r in filtered:
    env_type = get_cfg_env_type(r)
    mode = get_cfg_mode(r)
    seed = get_cfg_seed(r)
    key = (env_type, mode)

    groups.setdefault(key, {})
    if seed in [0, 1, 2] and seed not in groups[key]:
        groups[key][seed] = r

# ---- AGGREGATE ----
results = []
for (env_type, mode), by_seed in sorted(groups.items()):
    rs = list(by_seed.values())

    recs = []
    for r in rs:
        reward = get_last_value(r)
        ep_len = get_last_eval_step(r)
        nps = get_num_planning_steps(r)
        meta = get_meta_depth_counts(r)
        if reward is None or ep_len is None:
            continue
        recs.append((reward, ep_len, nps, meta))

    if not recs:
        continue

    rewards = np.array([x[0] for x in recs], float)
    eplens  = np.array([x[1] for x in recs], float)
    nps_arr = np.array([x[2] if x[2] is not None else np.nan for x in recs], float)
    meta_counts = {d: np.array([x[3][d] for x in recs], float) for d in [1, 2, 3, 4]}

    row = {
        "env_type": env_type,
        "mode": mode,
        "n_runs": len(recs),
        "reward_mean": float(rewards.mean()),
        "reward_std": float(rewards.std()),
        "ep_length_mean": float(eplens.mean()),
        "ep_length_std": float(eplens.std()),
        "num_planning_steps_mean": float(np.nanmean(nps_arr)) if np.any(~np.isnan(nps_arr)) else None,
        "num_planning_steps_std": float(np.nanstd(nps_arr)) if np.any(~np.isnan(nps_arr)) else None,
    }
    for d in [1, 2, 3, 4]:
        row[f"meta_depth_{d}_mean"] = float(meta_counts[d].mean())
        row[f"meta_depth_{d}_std"]  = float(meta_counts[d].std())

    results.append(row)

df = pd.DataFrame(results).sort_values(["env_type", "mode"]) if results else pd.DataFrame()
print(df)
df.to_csv("planning_percentage_0_05_by_env_and_mode.csv", index=False)

MATCHED: 147
          env_type    mode  n_runs   reward_mean    reward_std  \
0            Alien  reward       3    776.666667     32.998316   
1            Alien   value       3    920.000000    344.093011   
2           Amidar  reward       3    224.000000     19.646883   
3           Amidar   value       3    190.666667     47.849300   
4          Assault  reward       3    727.333333    170.826097   
5          Assault   value       3   1080.333333    245.180659   
6          Asterix  reward       3   4516.666667   2805.451043   
7          Asterix   value       3   5800.000000   2004.993766   
8        BankHeist  reward       3      3.333333      4.714045   
9        BankHeist   value       3      3.333333      4.714045   
10      BattleZone  reward       3   2333.333333   1885.618083   
11      BattleZone   value       3   1333.333333    471.404521   
12        Breakout  reward       3     55.333333     19.770910   
13        Breakout   value       3     89.666667     10.624918 